# Portfolio Multifractality Impact Analysis

This notebook extends `portfolio_mfdcca_var.ipynb` and `portfolio_robustness.ipynb`
with a comprehensive set of analyses that quantify:

1. **Individual-asset multifractality impact** on the portfolio
2. **Sector-level multifractality** comparisons & cross-correlations
3. **Portfolio-level extensions** — risk-budgeting, stress-tests, regime detection

All code reuses the same **TICKERS**, data-loading conventions, and
MF-DFA / MF-DCCA helper functions from those two companion notebooks.

### Parameters
* **q range:** −5 to 5, step 0.5
* **Scales:** 10 to 200 days (geometric spacing)
* **Visualisations:** static Matplotlib

## 0. Imports & Core Functions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import os
import warnings
from scipy.stats import kendalltau
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
from itertools import combinations

matplotlib.rcParams["figure.dpi"] = 110
warnings.filterwarnings('ignore', category=RuntimeWarning)

# ------------------------------------------------------------------
# MF-DFA / MF-DCCA core (same as portfolio_mfdcca_var.ipynb)
# ------------------------------------------------------------------
def profile(x):
    return np.cumsum(np.asarray(x) - np.mean(x))

def _local_residuals(Y, s, order=1, sign_filter=None):
    '''F2 per non-overlapping window (both directions). sign_filter in {None,"+","-"}
    keeps only windows whose local linear trend slope has that sign (asymmetric MF-DFA).'''
    N = len(Y); Ns = N // s; t = np.arange(s)
    out = []
    for v in range(Ns):
        seg = Y[v*s:(v+1)*s]
        c = np.polyfit(t, seg, order)
        if sign_filter == "+" and c[-2] < 0: continue
        if sign_filter == "-" and c[-2] >= 0: continue
        out.append(np.mean((seg - np.polyval(c, t)) ** 2))
    for v in range(Ns):
        seg = Y[N-(v+1)*s:N-v*s]
        c = np.polyfit(t, seg, order)
        if sign_filter == "+" and c[-2] < 0: continue
        if sign_filter == "-" and c[-2] >= 0: continue
        out.append(np.mean((seg - np.polyval(c, t)) ** 2))
    return np.array(out)

def mfdfa(x, q_list, scales, order=1, sign_filter=None):
    Y = profile(x)
    q_list = np.asarray(q_list, dtype=float)
    Fq = np.zeros((len(q_list), len(scales)))
    for si, s in enumerate(scales):
        F2 = _local_residuals(Y, s, order, sign_filter)
        F2 = F2[F2 > 1e-14]
        for qi, q in enumerate(q_list):
            if abs(q) < 1e-8:
                Fq[qi, si] = np.exp(0.5*np.mean(np.log(F2)))
            else:
                Fq[qi, si] = np.mean(F2**(q/2.0))**(1.0/q)
    hq = np.array([np.polyfit(np.log(scales), np.log(Fq[qi]), 1)[0]
                    for qi in range(len(q_list))])
    return Fq, hq

def multifractal_spectrum(q_list, hq):
    q = np.asarray(q_list, dtype=float)
    tau = q*hq - 1
    alpha = np.gradient(tau, q)
    f = q*alpha - tau
    return tau, alpha, f

def dcca_rho(x, y, s, order=1):
    X, Y2 = profile(x), profile(y)
    N = len(X); Ns = N // s; t = np.arange(s)
    Fxx = Fyy = Fxy = 0.0
    for v in range(Ns):
        xs, ys = X[v*s:(v+1)*s], Y2[v*s:(v+1)*s]
        cx = np.polyval(np.polyfit(t, xs, order), t)
        cy = np.polyval(np.polyfit(t, ys, order), t)
        rx, ry = xs-cx, ys-cy
        Fxx += np.mean(rx**2); Fyy += np.mean(ry**2); Fxy += np.mean(rx*ry)
    Fxx, Fyy, Fxy = Fxx/Ns, Fyy/Ns, Fxy/Ns
    return Fxy / np.sqrt(Fxx*Fyy)

def mf_dcca_hxy(x, y, q_list, scales, order=1):
    X, Y2 = profile(x), profile(y)
    q_list = np.asarray(q_list, dtype=float)
    Fxy_q = np.zeros((len(q_list), len(scales)))
    for si, s in enumerate(scales):
        N = len(X); Ns = N // s; t = np.arange(s)
        Fxy = np.empty(2*Ns)
        for v in range(Ns):
            xs, ys = X[v*s:(v+1)*s], Y2[v*s:(v+1)*s]
            cx = np.polyval(np.polyfit(t, xs, order), t)
            cy = np.polyval(np.polyfit(t, ys, order), t)
            Fxy[v] = np.mean((xs-cx)*(ys-cy))
        for v in range(Ns):
            xs = X[N-(v+1)*s:N-v*s]; ys = Y2[N-(v+1)*s:N-v*s]
            cx = np.polyval(np.polyfit(t, xs, order), t)
            cy = np.polyval(np.polyfit(t, ys, order), t)
            Fxy[Ns+v] = np.mean((xs-cx)*(ys-cy))
        Fxy = np.abs(Fxy); Fxy = Fxy[Fxy > 1e-14]
        for qi, q in enumerate(q_list):
            Fxy_q[qi, si] = np.mean(Fxy**(q/2.0))**(1.0/q) if abs(q) > 1e-8 \
                             else np.exp(0.5*np.mean(np.log(Fxy)))
    return np.array([np.polyfit(np.log(scales), np.log(Fxy_q[qi]), 1)[0]
                      for qi in range(len(q_list))])

def hill_estimator(losses, k):
    '''Hill estimator of the tail index alpha from the k largest losses.'''
    x = np.sort(losses)[::-1][:k+1]
    logs = np.log(x[:-1] / x[-1])
    return k / np.sum(logs)

def shuffle_surrogate(x, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    s = x.copy()
    rng.shuffle(s)
    return s

print("Core MF-DFA / MF-DCCA / Hill / surrogate functions loaded.")

## 1. Data Loading

Uses the exact same `TICKERS` list and `stock_csv_data/` convention as the
companion notebooks.

In [ ]:
TICKERS = [
    # US Large Cap Tech
    "AAPL", "MSFT", "GOOG", "AMZN", "NVDA",
    "META", "TSLA", "NFLX", "AMD", "INTC",
    # US Financials
    "JPM", "BAC", "GS", "MS", "WFC",
    "BLK", "C", "AXP", "V", "MA",
    # US Healthcare & Consumer
    "JNJ", "PFE", "UNH", "MRK", "ABBV",
    "PG", "KO", "PEP", "WMT", "COST",
    # US Energy & Industrials
    "XOM", "CVX", "COP", "BA", "CAT",
    "GE", "HON", "LMT", "RTX", "UPS",
    # Commodities & Crypto
    "GC=F", "SI=F", "CL=F", "BTC-USD", "ETH-USD",
    # International ETFs
    "EWJ", "EWZ", "FXI", "EWG", "EWU",
]

START, END = "2010-01-01", "2026-04-11"
OUT_DIR = "stock_csv_data"
os.makedirs(OUT_DIR, exist_ok=True)

# Sector-diverse sub-portfolio
PORTFOLIO = ["AAPL", "JPM", "XOM", "JNJ", "GC=F", "BTC-USD", "EWJ"]
SECTOR    = {"AAPL":"Tech", "JPM":"Financials", "XOM":"Energy", "JNJ":"Healthcare",
             "GC=F":"Commodity", "BTC-USD":"Crypto", "EWJ":"Intl ETF"}

def csv_path(ticker, folder=OUT_DIR):
    return os.path.join(folder, f"{ticker.replace('=', '_')}_data.csv")

def get_prices(tickers, start=START, end=END, folder=OUT_DIR):
    series = {}
    for t in tickers:
        path = csv_path(t, folder)
        if not os.path.exists(path):
            import yfinance as yf
            print(f"Downloading {t} ...")
            data = yf.download(t, start=start, end=end, progress=False)
            if isinstance(data.columns, pd.MultiIndex):
                data.columns = data.columns.get_level_values(0)
            data.to_csv(path)
        df = pd.read_csv(path)
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"]).set_index("Date")
        series[t] = pd.to_numeric(df["Close"], errors="coerce")
    price_df = pd.DataFrame(series).dropna()
    return price_df

price_df = get_prices(PORTFOLIO)
returns  = {t: np.diff(np.log(price_df[t].values)) for t in PORTFOLIO}
dates    = price_df.index[1:]
N        = len(dates)
print(f"{len(PORTFOLIO)} assets, {N} aligned daily log-returns "
      f"({dates[0].date()} -> {dates[-1].date()}, ~{N/252:.1f} yrs)")
price_df.tail()

## 2. Baseline MF-DFA per Component & Portfolio

Compute **h(q)**, **spectral width Δh**, and **multifractal spectrum (τ, α, f)**
for every asset and for the equal-weight portfolio.

In [ ]:
q_list = np.arange(-5, 5.5, 0.5)
scales = np.unique(np.geomspace(10, 200, 20).astype(int))

weights = np.ones(len(PORTFOLIO)) / len(PORTFOLIO)

# ── Per-component MF-DFA ──
spectra = {}
h_matrix = []   # shape: (n_assets, n_q)
for name in PORTFOLIO:
    r = returns[name]
    Fq, hq = mfdfa(r, q_list, scales)
    tau, alpha, f = multifractal_spectrum(q_list, hq)
    spectra[name] = dict(hq=hq, tau=tau, alpha=alpha, f=f,
                         width=hq[0]-hq[-1], Fq=Fq)
    h_matrix.append(hq)

h_matrix = np.array(h_matrix)   # (n_assets, n_q)

# Print summary
for i, name in enumerate(PORTFOLIO):
    s = spectra[name]
    print(f"{name:9s} ({SECTOR[name]:10s})  h(2)={s['hq'][q_list==2][0]:.3f}"
          f"   width Δh={s['width']:.3f}")

# ── Equal-weight portfolio MF-DFA ──
port_ret = sum(w*returns[n] for w, n in zip(weights, PORTFOLIO))
Fq_p, hq_p = mfdfa(port_ret, q_list, scales)
tau_p, alpha_p, f_p = multifractal_spectrum(q_list, hq_p)
width_p     = hq_p[0] - hq_p[-1]
naive_width = sum(w*spectra[n]["width"] for w, n in zip(weights, PORTFOLIO))

# ── MDR ──
MDR = naive_width / width_p if abs(width_p) > 1e-12 else np.nan
print(f"\nPortfolio Δh = {width_p:.4f},  naive avg Δh = {naive_width:.4f}")
print(f"MDR = {MDR:.3f}")

## 3. Individual-Asset Multifractality Impact

### 3a. Weighted-Average Hurst h(q)
Shows how each security pushes the portfolio's scaling exponent away from 0.5.

In [ ]:
# Weighted Hurst contribution
h_weighted = (weights[:, None] * h_matrix).sum(axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
for i, name in enumerate(PORTFOLIO):
    ax.plot(q_list, h_matrix[i], label=f"{name} ({SECTOR[name]})", alpha=0.7)
ax.plot(q_list, h_weighted, 'k--', lw=2.5, label='Portfolio (w-avg)')
ax.plot(q_list, hq_p, 'r-', lw=2.5, label='Portfolio MF-DFA')
ax.axhline(0.5, color='gray', ls=':', lw=1)
ax.set_xlabel('q'); ax.set_ylabel('h(q)')
ax.set_title('3a. Weighted Hurst Exponents vs. q')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

### 3b. Multifractal-aware Diversification Ratio (MDR)
Analogous to the classic diversification ratio but uses MF-DFA spectral width.

In [ ]:
spectral_widths = np.array([spectra[n]["width"] for n in PORTFOLIO])
sigma_portfolio_mf = width_p

print("=== Multifractal Diversification Ratio (MDR) ===")
print(f"Naive weighted spectral width : {naive_width:.4f}")
print(f"Portfolio spectral width       : {width_p:.4f}")
print(f"MDR = {MDR:.3f}")
print()

# Per-asset contribution to MDR
print(f"{'Ticker':9s} {'Sector':12s} {'w_i':>6s} {'Δh_i':>8s} {'w·Δh':>8s} {'Contrib%':>8s}")
for i, name in enumerate(PORTFOLIO):
    w = weights[i]
    dh = spectral_widths[i]
    contrib = w * dh
    pct = 100.0 * contrib / naive_width if naive_width else 0
    print(f"{name:9s} {SECTOR[name]:12s} {w:6.3f} {dh:8.3f} {contrib:8.4f} {pct:7.1f}%")

### 3c. Contribution-to-Portfolio Tail Risk (CTR)
Quantifies which assets dominate left-tail heaviness using α from negative-q side.

In [ ]:
neg_q_mask = q_list < 0
print("=== Contribution to Tail Risk ===")
print(f"{'Ticker':9s} {'Sector':12s} {'α_neg (avg)':>12s} {'w·α_neg':>10s} {'CTR%':>8s}")

alpha_neg_vals = []
for name in PORTFOLIO:
    alpha_neg_vals.append(np.mean(spectra[name]["alpha"][neg_q_mask]))

alpha_neg_vals = np.array(alpha_neg_vals)
w_alpha = weights * alpha_neg_vals
ctr = w_alpha / w_alpha.sum() * 100

for i, name in enumerate(PORTFOLIO):
    print(f"{name:9s} {SECTOR[name]:12s} {alpha_neg_vals[i]:12.4f} "
          f"{w_alpha[i]:10.4f} {ctr[i]:7.1f}%")

### 3d. Asymmetric MF-DFA — Up vs Down Persistence
Visualises the leverage effect at the fractal level using `sign_filter`.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True)
axes = axes.flatten()

for i, name in enumerate(PORTFOLIO):
    ax = axes[i]
    r = returns[name]
    _, hq_up   = mfdfa(r, q_list, scales, sign_filter='+')
    _, hq_down = mfdfa(r, q_list, scales, sign_filter='-')
    _, hq_all  = mfdfa(r, q_list, scales)
    ax.plot(q_list, hq_up,   'g-', lw=1.5, label='Up (h⁺)')
    ax.plot(q_list, hq_down, 'r-', lw=1.5, label='Down (h⁻)')
    ax.plot(q_list, hq_all,  'k--', lw=1, alpha=0.5, label='All')
    ax.set_title(f"{name} ({SECTOR[name]})", fontsize=9)
    ax.set_xlabel('q'); ax.set_ylabel('h(q)')
    if i == 0: ax.legend(fontsize=7)

# hide unused subplot
if len(PORTFOLIO) < len(axes):
    for j in range(len(PORTFOLIO), len(axes)):
        axes[j].set_visible(False)

plt.suptitle('3d. Asymmetric MF-DFA — Up vs Down Move Persistence', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

### 3e. Heat-map of Δ-Hurst × Weight
Instantly spots assets that increase or decrease portfolio multifractality.

In [ ]:
delta_h = h_matrix - hq_p[None, :]   # (n_assets, n_q)
impact  = delta_h * weights[:, None]  # weighted delta

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(impact, aspect='auto', cmap='RdBu_r',
               extent=[q_list[0], q_list[-1], len(PORTFOLIO)-0.5, -0.5])
ax.set_yticks(range(len(PORTFOLIO)))
ax.set_yticklabels([f"{n} ({SECTOR[n]})" for n in PORTFOLIO])
ax.set_xlabel('q')
ax.set_title('3e. Δh·w  Heat-map (red = increases multifractality)')
plt.colorbar(im, ax=ax, label='Δh × weight')
plt.tight_layout(); plt.show()

### 3f. Summary Table

In [ ]:
summary_rows = []
for i, name in enumerate(PORTFOLIO):
    s = spectra[name]
    h2 = s['hq'][q_list == 2][0]
    dh = s['width']
    mdr_c = weights[i] * dh / naive_width * 100 if naive_width else 0
    summary_rows.append({
        'Ticker': name, 'Sector': SECTOR[name],
        'Weight': weights[i], 'H(2)': h2,
        'Δh': dh, 'Δh·w': weights[i]*dh,
        'MDR_contrib%': mdr_c, 'CTR%': ctr[i]
    })

summary_df = pd.DataFrame(summary_rows).set_index('Ticker')
display(summary_df.round(4))

## 4. Sector-Level Multifractality

### 4a. Sector-Averaged H(q) & τ(q)

In [ ]:
# Group tickers by sector
sector_groups = {}
for name in PORTFOLIO:
    sec = SECTOR[name]
    sector_groups.setdefault(sec, []).append(name)

sector_h = {}
for sec, tickers in sector_groups.items():
    indices = [PORTFOLIO.index(t) for t in tickers]
    sector_h[sec] = h_matrix[indices].mean(axis=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for sec, hq_s in sector_h.items():
    ax1.plot(q_list, hq_s, 'o-', ms=4, label=sec)
    tau_s = q_list * hq_s - 1
    ax2.plot(q_list, tau_s, 'o-', ms=4, label=sec)

ax1.axhline(0.5, color='gray', ls=':', lw=1)
ax1.set_xlabel('q'); ax1.set_ylabel('H(q)')
ax1.set_title('4a. Sector-Averaged Hurst Exponent')
ax1.legend(fontsize=8)

ax2.set_xlabel('q'); ax2.set_ylabel('τ(q)')
ax2.set_title('4a. Sector-Averaged Renyi Exponent τ(q)')
ax2.legend(fontsize=8)

plt.tight_layout(); plt.show()

### 4b. Sector-Pairwise MF-DCCA Correlation Matrix

In [ ]:
s_fixed = np.unique(np.geomspace(16, N//8, 12).astype(int))
sector_names = sorted(sector_groups.keys())
n_sec = len(sector_names)
rho_matrix = np.eye(n_sec)

for i in range(n_sec):
    for j in range(i+1, n_sec):
        # use returns of first ticker in each sector as representative
        ti = sector_groups[sector_names[i]][0]
        tj = sector_groups[sector_names[j]][0]
        rhos = [dcca_rho(returns[ti], returns[tj], s) for s in s_fixed]
        avg_rho = np.mean(rhos)
        rho_matrix[i, j] = avg_rho
        rho_matrix[j, i] = avg_rho

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(rho_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(n_sec)); ax.set_xticklabels(sector_names, rotation=45, ha='right')
ax.set_yticks(range(n_sec)); ax.set_yticklabels(sector_names)
for i in range(n_sec):
    for j in range(n_sec):
        ax.text(j, i, f"{rho_matrix[i,j]:.2f}", ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, label='ρ_DCCA')
ax.set_title('4b. Sector-Pairwise MF-DCCA Correlation')
plt.tight_layout(); plt.show()

### 4c. Cluster Dendrogram on MF-DCCA Distances

In [ ]:
d_matrix = 1 - rho_matrix
np.fill_diagonal(d_matrix, 0)
d_matrix = np.maximum(d_matrix, 0)
condensed = squareform(d_matrix)

Z = linkage(condensed, method='average')

fig, ax = plt.subplots(figsize=(8, 5))
dendrogram(Z, labels=sector_names, ax=ax, leaf_rotation=45)
ax.set_ylabel('Distance (1 − ρ_DCCA)')
ax.set_title('4c. Multifractal Similarity Dendrogram of Sectors')
plt.tight_layout(); plt.show()

### 4d. Sector-Specific Tail Index via Hill Estimator

In [ ]:
k_hill = 50  # top-k losses for Hill estimator

print(f"{'Sector':12s} {'Hill α':>8s}")
for sec in sector_names:
    tickers_s = sector_groups[sec]
    # aggregate sector returns (equal weight within sector)
    sec_ret = np.mean([returns[t] for t in tickers_s], axis=0)
    losses = -sec_ret
    losses = losses[losses > 0]
    k = min(k_hill, len(losses) - 1)
    alpha_hill = hill_estimator(losses, k)
    print(f"{sec:12s} {alpha_hill:8.3f}")

## 5. Volatility-Character Retention & DCCA Rankings

In [ ]:
vol   = {name: np.abs(r) for name, r in returns.items()}
vol_p = np.abs(port_ret)

vol_dcca_rows = []
for name in PORTFOLIO:
    rhos = [dcca_rho(vol[name], vol_p, s) for s in s_fixed]
    vol_dcca_rows.append((name, np.mean(rhos), rhos))
vol_dcca_rows.sort(key=lambda r: -r[1])

print("Volatility-character retention ranking (avg ρ_DCCA):")
for rank, (name, avg_rho, _) in enumerate(vol_dcca_rows, 1):
    print(f"  {rank}. {name:9s} ({SECTOR[name]:10s})  avg ρ_DCCA = {avg_rho:.3f}")

# Linear rolling-vol correlation comparison
ROLL_VOL_WIN = 21
roll_vol   = {name: pd.Series(vol[name]).rolling(ROLL_VOL_WIN).mean().values for name in PORTFOLIO}
roll_vol_p = pd.Series(vol_p).rolling(ROLL_VOL_WIN).mean().values

print(f"\n{'Ticker':9s}  corr(rolling {ROLL_VOL_WIN}d |r_i|, rolling |r_portfolio|)")
corr_rows = []
for name in PORTFOLIO:
    mask = ~(np.isnan(roll_vol[name]) | np.isnan(roll_vol_p))
    c = np.corrcoef(roll_vol[name][mask], roll_vol_p[mask])[0,1]
    corr_rows.append((name, c))
corr_rows.sort(key=lambda r: -r[1])
for name, c in corr_rows:
    print(f"{name:9s}  {c:.3f}")

## 6. Rolling Multifractal Diversification Ratio (MDR)

In [ ]:
ROLL_WINDOW = 252  # 1-year rolling window
q_roll = np.array([-5, -3, -1, 1, 2, 3, 5], dtype=float)
scales_roll = np.unique(np.geomspace(10, ROLL_WINDOW//4, 10).astype(int))

mdr_series = []
mdr_dates  = []

for t_start in range(0, N - ROLL_WINDOW, 21):  # step every ~month for speed
    t_end = t_start + ROLL_WINDOW
    window_returns = {name: returns[name][t_start:t_end] for name in PORTFOLIO}

    # Per-component widths
    widths_w = []
    for name in PORTFOLIO:
        _, hq_w = mfdfa(window_returns[name], q_roll, scales_roll)
        widths_w.append(hq_w[0] - hq_w[-1])
    widths_w = np.array(widths_w)

    # Portfolio width
    port_w = sum(w*window_returns[n] for w, n in zip(weights, PORTFOLIO))
    _, hq_pw = mfdfa(port_w, q_roll, scales_roll)
    width_pw = hq_pw[0] - hq_pw[-1]

    naive_w = (weights * widths_w).sum()
    mdr_val = naive_w / width_pw if abs(width_pw) > 1e-12 else np.nan
    mdr_series.append(mdr_val)
    mdr_dates.append(dates[t_end - 1])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(mdr_dates, mdr_series, 'b-', lw=1.5)
ax.axhline(1.0, color='gray', ls=':', lw=1)
ax.set_xlabel('Date'); ax.set_ylabel('MDR')
ax.set_title('6. Rolling Multifractal Diversification Ratio (1-year window)')
plt.tight_layout(); plt.show()

## 7. Multifractal-Aware Risk Budget

Approximate ∂σ_portfolio / ∂w_i using finite differences on the MF-DFA spectral width.

In [ ]:
delta_w = 0.01

def portfolio_spectral_width(w_vec):
    pr = sum(w*returns[n] for w, n in zip(w_vec, PORTFOLIO))
    _, hq_tmp = mfdfa(pr, q_list, scales)
    return hq_tmp[0] - hq_tmp[-1]

base_width = portfolio_spectral_width(weights)

print("=== Multifractal-Aware Risk Budget ===")
print(f"{'Ticker':9s} {'Sector':12s} {'∂σ/∂w':>10s} {'RB_i':>10s} {'RB%':>8s}")
grad = np.zeros(len(PORTFOLIO))
for i, name in enumerate(PORTFOLIO):
    w_up = weights.copy(); w_up[i] += delta_w
    w_up /= w_up.sum()
    grad[i] = (portfolio_spectral_width(w_up) - base_width) / delta_w

risk_budget = weights * grad
rb_total = risk_budget.sum()

for i, name in enumerate(PORTFOLIO):
    pct = 100.0 * risk_budget[i] / rb_total if rb_total else 0
    print(f"{name:9s} {SECTOR[name]:12s} {grad[i]:10.4f} {risk_budget[i]:10.4f} {pct:7.1f}%")

## 8. Multifractal-Adjusted Expected Shortfall

Scale ES by the ratio of MF-DFA-implied tail heaviness to monofractal baseline.

In [ ]:
alpha_es = 0.05
losses = -port_ret
sorted_losses = np.sort(losses)[::-1]
n_tail = max(int(len(losses) * alpha_es), 1)

ES_classical = np.mean(sorted_losses[:n_tail])

# Tail index from portfolio MF-DFA — use h(q=-5) as proxy for tail persistence
h_tail = hq_p[q_list == -5][0] if any(q_list == -5) else hq_p[0]
h_mono = 0.5  # monofractal baseline

# Scaling factor: higher h_tail => more persistent tail => higher ES
scaling = h_tail / h_mono if h_mono > 0 else 1.0
ES_adjusted = ES_classical * scaling

print("=== Multifractal-Adjusted Expected Shortfall ===")
print(f"Classical ES ({alpha_es*100:.0f}%): {ES_classical:.6f}")
print(f"Portfolio h(q=-5):          {h_tail:.4f}")
print(f"Monofractal baseline h:     {h_mono:.4f}")
print(f"Scaling factor:             {scaling:.4f}")
print(f"MF-Adjusted ES:             {ES_adjusted:.6f}")

## 9. Findings Summary

> **Run the notebook** to populate these findings with actual computed values.

### Individual-Asset Impact
- The **weighted Hurst plot** (§3a) reveals how each asset pushes the portfolio's
  scaling exponent; crypto/energy tend to have higher persistence (h > 0.5).
- The **MDR table** (§3b) shows whether diversification reduces or amplifies
  multifractality — MDR > 1 means diversification helps.
- **CTR** (§3c) pinpoints which assets dominate tail risk from the fractal perspective.
- **Asymmetric MF-DFA** (§3d) exposes leverage effects: assets whose h⁻(q) exceeds
  h⁺(q) are more persistent on the downside.
- The **Δh·w heat-map** (§3e) visually highlights portfolio multifractality contributors.

### Sector-Level Insights
- **Sector-averaged H(q)** (§4a) enables direct comparison of persistence regimes
  across sectors.
- The **MF-DCCA matrix & dendrogram** (§4b-c) reveal cross-scale contagion paths.
- **Hill tail indices** (§4d) link multifractality to extreme-loss behaviour per sector.

### Portfolio Extensions
- The **rolling MDR** (§6) time-series flags regime transitions where fractal
  diversification gains expand or collapse.
- **Risk-budget** (§7) allocates MF-DFA sensitivity across assets.
- **MF-Adjusted ES** (§8) scales classical ES by empirical tail persistence.